# Exploring Chat Templates with SmolLM2

This notebook demonstrates how to use chat templates with the `SmolLM2` model.
Key concepts covered:
- Loading and configuring a language model for chat
- Formatting messages using chat templates
- Understanding tokenization of chat messages

In [ ]:
# Install the requirements in Google Colab
# !pip install transformers datasets trl huggingface_hub

# Authenticate to Hugging Face
from huggingface_hub import login

login()
# for convenience you can create an environment variable containing your hub token as HF_TOKEN

In [ ]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer # Core Hugging Face components for model and tokenizer
from trl import setup_chat_format # Helper for chat formatting
import torch # Deep learning framework

## Set up device and model

In [ ]:
# First determine best available hardware
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the base model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Configure the model and tokenizer for chat format
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

## SmolLM2 Chat Template

Chat templates help structure interactions between users and AI models, ensuring consistent and contextually appropriate responses.

Let's explore how to use a chat template with the `SmolLM2` model. We'll define a simple conversation and apply the chat template.

In [ ]:
# Define messages for SmolLM2
"""
Chat messages are structured as a list of dictionaries.
Each message has:
- role: either 'user' or 'assistant'
- content: the actual message text
"""

messages = [
    {"role": "user", "content": "Hello, how are you?"},
    {
        "role": "assistant",
        "content": "I'm doing well, thank you! How can I assist you today?",
    },
]

# Apply chat template without tokenization

The tokenizer represents the conversation as a string with special tokens to describe the role of the user and the assistant.

The chat template adds special tokens to structure the conversation:
- <|im_start|> and <|im_end|> mark message boundaries
- The role (user/assistant) is included before each message
This helps the model understand the conversation flow.

In [ ]:
input_text = tokenizer.apply_chat_template(messages, tokenize=False)

print("Conversation with template:", input_text)

Conversation with template: <|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you! How can I assist you today?<|im_end|>



# Decode the conversation
When preparing for model input, we:
1. Apply the chat template
2. Tokenize the text
3. Add a generation prompt for the model's response

Note that the conversation is represented as above but with a further assistant message.

In [ ]:
input_text = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True
)

print("Conversation decoded:", tokenizer.decode(token_ids=input_text))

Conversation decoded: <|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing well, thank you! How can I assist you today?<|im_end|>
<|im_start|>assistant



# Tokenize the conversation

Of course, the tokenizer also tokenizes the conversation and special token as ids that relate to the model's vocabulary.

The model processes text as token IDs - numbers that represent words/subwords.
This shows the actual input format the model receives.


In [ ]:
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

print("Conversation tokenized:", input_text)

Conversation tokenized: [1, 4093, 198, 19556, 28, 638, 359, 346, 47, 2, 198, 1, 520, 9531, 198, 57, 5248, 2567, 876, 28, 9984, 346, 17, 1073, 416, 339, 4237, 346, 1834, 47, 2, 198, 1, 520, 9531, 198]


## Exercise: Processing Datasets for Supervised Fine-Tuning (SFT)

### Overview
In this exercise, we'll learn how to prepare a dataset for supervised fine-tuning of language models.
We'll work with two datasets of different complexity:

🐢 **Beginner**: `HuggingFaceTB/smoltalk` - A simple conversational dataset
🐕 **Advanced**: `openai/gsm8k` - A mathematical reasoning dataset

### What We'll Learn
- Loading datasets from Hugging Face Hub
- Converting raw data into chat format
- Applying chat templates for model training
- Processing and validating the formatted data

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Process a dataset for SFT</h2>
    <p>Take a dataset from the Hugging Face hub and process it for SFT. </p>
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Convert the `HuggingFaceTB/smoltalk` dataset into chatml format.</p>
    <p>🐕 Convert the `openai/gsm8k` dataset into chatml format.</p>
</div>

In [ ]:
from IPython.core.display import display, HTML

display(
    HTML(
        """<iframe
  src="https://huggingface.co/datasets/HuggingFaceTB/smoltalk/embed/viewer/all/train?row=0"
  frameborder="0"
  width="100%"
  height="360px"
></iframe>
"""
    )
)

### 🐢 Beginner Exercise: Processing SmolTalk Dataset

In [ ]:
from datasets import load_dataset

# Load the dataset
ds = load_dataset("HuggingFaceTB/smoltalk", "everyday-conversations")

def process_smoltalk_dataset(sample):
    """
    Convert SmolTalk format to chat format.

    Input format:
    {
        "conversation": "User: Hello\nAssistant: Hi there!"
    }

    Output format:
    {
        "messages": [
            {"role": "user", "content": "Hello"},
            {"role": "assistant", "content": "Hi there!"}
        ]
    }
    """
    # Split conversation into turns
    turns = sample["conversation"].split("\n")

    # Convert turns into messages
    messages = []
    for turn in turns:
        role, content = turn.split(": ", 1)
        role = role.lower()  # normalize role to lowercase
        messages.append({"role": role, "content": content})

    # Apply chat template
    formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False)

    return {"formatted_text": formatted_chat}

# Process the dataset
processed_ds = ds.map(process_smoltalk_dataset)

# Show example
print("Original format:")
print(ds[0]["conversation"])
print("\nProcessed format:")
print(processed_ds[0]["formatted_text"])

In [ ]:
display(
    HTML(
        """<iframe
  src="https://huggingface.co/datasets/openai/gsm8k/embed/viewer/main/train"
  frameborder="0"
  width="100%"
  height="360px"
></iframe>
"""
    )
)

### 🐕 Advanced Exercise: Processing GSM8K Dataset

The GSM8K dataset contains math word problems with step-by-step solutions.
This requires:
1. Formatting the problem as a user question
2. Structuring the solution as an assistant response
3. Preserving the step-by-step reasoning

In [ ]:
# Load GSM8K dataset
ds = load_dataset("openai/gsm8k", "main")

def process_gsm8k_dataset(sample):
    """
    Convert GSM8K format to chat format.

    Input format:
    {
        "question": "Math problem...",
        "answer": "Let's solve step by step...\n####\n42"
    }

    Output format:
    {
        "messages": [
            {"role": "user", "content": "Solve this math problem: ..."},
            {"role": "assistant", "content": "Let me solve this step by step..."}
        ]
    }
    """
    # Format question as user message
    user_msg = {"role": "user", "content": f"Solve this math problem: {sample['question']}"}

    # Format answer as assistant message
    # Remove the final answer marker and number
    solution = sample["answer"].split("####")[0].strip()
    assistant_msg = {"role": "assistant", "content": solution}

    messages = [user_msg, assistant_msg]

    # Apply chat template
    formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False)

    return {"formatted_text": formatted_chat}

# Process the dataset
processed_ds = ds.map(process_gsm8k_dataset)

# Show example
print("Original format:")
print(f"Question: {ds['train'][0]['question']}")
print(f"Answer: {ds['train'][0]['answer']}")
print("\nProcessed format:")
print(processed_ds['train'][0]["formatted_text"])

### Key Learning Points

1. **Dataset Structure**: Different datasets require different processing approaches
   - SmolTalk: Simple conversation parsing
   - GSM8K: Complex problem-solution formatting

2. **Chat Templates**: The template adds special tokens and structure
   - Marks message boundaries
   - Identifies speaker roles
   - Maintains conversation flow

3. **Data Validation**: Always check your processed output
   - Verify format matches model expectations
   - Ensure no information is lost
   - Test with different examples

Try modifying the processing functions to handle different dataset formats or add additional features like:
- Input validation
- Error handling
- Custom formatting options

## Conclusion

This notebook demonstrated how to apply chat templates to different models, `SmolLM2`. By structuring interactions with chat templates, we can ensure that AI models provide consistent and contextually relevant responses.

In the exercise you tried out converting a dataset into chatml format. Luckily, TRL will do this for you, but it's useful to understand what's going on under the hood.